In [1]:
import pandas as pd

label_columns = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
    'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
]

def load_and_clean_chexpert(csv_path):
    # Load metadata
    df = pd.read_csv(csv_path)
    
    # Ensure numeric types and handle NaNs
    df[label_columns] = df[label_columns].apply(pd.to_numeric, errors='coerce').fillna(0).astype(float)
    
    # Apply U-Zeros policy: Replace all -1 (uncertain) with 0 (negative)
    df[label_columns] = df[label_columns].replace(-1.0, 0.0)
    
    # Standardize image paths for the Kaggle environment
    df['Path'] = df['Path'].str.replace('CheXpert-v1.0-small/', '', regex=False)
    df['Path'] = df['Path'].apply(lambda x: f"/kaggle/input/chexpert/{x}")
    
    # Return cleaned dataframe containing only the Path and the 14 labels
    return pd.concat([df['Path'], df[label_columns]], axis=1)

# Generate cleanly separated dataframes
train_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/train.csv')
valid_df = load_and_clean_chexpert('/kaggle/input/datasets/ashery/chexpert/valid.csv')

# Create the 5% subset for your convergence test
sampled_train_df = train_df.sample(frac=0.05, random_state=42).reset_index(drop=True)

print(f"Full Training Set: {len(train_df)}")
print(f"Sampled Training Set (5%): {len(sampled_train_df)}")
print(f"Validation Set: {len(valid_df)}")

Full Training Set: 223414
Sampled Training Set (5%): 11171
Validation Set: 234


In [2]:
import os
import torch
import cv2
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms

class CheXpertDistillationDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe
        
        # 1. Spatial transforms (Applied once so both models see the exact same geometry)
        self.spatial_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            # Add random spatial augmentations here if needed (e.g., RandomHorizontalFlip)
        ])
        
        # 2. EfficientNet Normalization (ImageNet stats)
        self.effnet_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # 3. ViT Normalization ([-1, 1] stats)
        self.vit_normalize = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        raw_path = row['Path']
        
        if raw_path.startswith('CheXpert-v1.0-small/'):
            relative_path = raw_path.replace('CheXpert-v1.0-small/', '')
        elif raw_path.startswith('/kaggle/input/chexpert/'):
            relative_path = raw_path.replace('/kaggle/input/chexpert/', '')
        else:
            relative_path = raw_path
            
        relative_path = relative_path.lstrip('/')
        image_path = os.path.join('/kaggle/input/datasets/ashery/chexpert/', relative_path)
        
        # OpenCV memory fragmentation fix
        img_array = cv2.imread(image_path)
        img_array = cv2.cvtColor(img_array, cv2.COLOR_BGR2RGB)
        image = Image.fromarray(img_array)
        
        label = torch.tensor(row[1:].values.astype(float), dtype=torch.float32)
        
        # Apply common geometry
        base_image = self.spatial_transform(image)
        
        # Branch normalizations natively
        effnet_image = self.effnet_normalize(base_image)
        vit_image = self.vit_normalize(base_image)
        
        return effnet_image, vit_image, label

# Initialize single datasets that handle both normalizations natively
train_dataset = CheXpertDistillationDataset(dataframe=train_df)
valid_dataset = CheXpertDistillationDataset(dataframe=valid_df)

In [3]:
import torch
import torch.nn as nn

class PCAProjector(nn.Module):
    def __init__(self, cnn_channels, vit_dim):
        super().__init__()
        # 3x3 convolutions to map CNN features to Attention Space
        self.conv_q = nn.Conv2d(cnn_channels, vit_dim, kernel_size=3, padding=1)
        self.conv_k = nn.Conv2d(cnn_channels, vit_dim, kernel_size=3, padding=1)
        self.conv_v = nn.Conv2d(cnn_channels, vit_dim, kernel_size=3, padding=1)
        self.scale = vit_dim ** -0.5

    def forward(self, student_feats, teacher_q, teacher_k, teacher_v, p=0.5):
        # Flatten CNN spatial dimensions: [Batch, Channels, Height, Width] -> [Batch, Seq_Len, Channels]
        B, C, H, W = student_feats.shape
        
        q_s = self.conv_q(student_feats).flatten(2).transpose(1, 2)
        k_s = self.conv_k(student_feats).flatten(2).transpose(1, 2)
        v_s = self.conv_v(student_feats).flatten(2).transpose(1, 2)

        # The g(.) function: Uniform distribution probability mask
        # True means replace with Teacher, False means keep Student
        mask_q = (torch.rand_like(q_s) > p).float()
        mask_k = (torch.rand_like(k_s) > p).float()
        mask_v = (torch.rand_like(v_s) > p).float()

        # Element-wise stochastic substitution
        q_mixed = mask_q * teacher_q + (1.0 - mask_q) * q_s
        k_mixed = mask_k * teacher_k + (1.0 - mask_k) * k_s
        v_mixed = mask_v * teacher_v + (1.0 - mask_v) * v_s

        # Compute mixed attention map
        attn_scores = (q_mixed @ k_mixed.transpose(-2, -1)) * self.scale
        mixed_attn_map = attn_scores.softmax(dim=-1)
        
        # Final projected representation
        projected_output = mixed_attn_map @ v_mixed

        return projected_output, mixed_attn_map, v_mixed

In [4]:
class GLProjector(nn.Module):
    def __init__(self, cnn_channels, vit_dim, num_groups=49):
        super().__init__()
        # We use 49 groups assuming a 2x2 neighborhood on a 14x14 grid
        # Conv1d with groups acts as independent FC layers applied to isolated chunks
        self.group_fc = nn.Conv1d(
            in_channels=cnn_channels * num_groups, 
            out_channels=vit_dim * num_groups, 
            kernel_size=1, 
            groups=num_groups
        )
        self.dropout = nn.Dropout(0.2)

    def forward(self, student_feats):
        # student_feats shape: [Batch, Channels, 14, 14]
        B, C, H, W = student_feats.shape
        
        # Flatten spatial dims to sequence length [Batch, Channels, 196]
        x = student_feats.view(B, C, H * W)
        
        # Reshape to group the spatial tokens for the independent FC layers
        # [Batch, Channels * Groups, Pixels_per_Group]
        num_groups = self.group_fc.groups
        pixels_per_group = (H * W) // num_groups
        x = x.view(B, C * num_groups, pixels_per_group)

        # Apply shared-weight FC per group with dropout
        x = self.dropout(self.group_fc(x))

        # Reconstruct to match ViT dimensions [Batch, 196, vit_dim]
        x = x.view(B, num_groups * pixels_per_group, -1)
        
        return x

In [5]:
from torch.utils.data import DataLoader

# train_dataset = CheXpertDataset(dataframe=dataset, transform=transform)
# valid_dataset = CheXpertDataset(dataframe=dataset_valid, transform=transform)

# Drastically increase batch size to saturate VRAM and reduce transfer overhead
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE,              # Safe for GPU VRAM
    shuffle=True, 
    num_workers=0,               
    pin_memory=False,            
)

valid_loader = DataLoader(
    valid_dataset, 
    batch_size=BATCH_SIZE,       
    shuffle=False, 
    num_workers=0,               
    pin_memory=False             
)
# Example: Iterate over the DataLoader
for eff_images, vit_images, labels in train_loader:
    print("EfficientNET Image batch shape:", eff_images.shape)
    print("ViT Image batch shape:", vit_images.shape)
    print("Label batch shape:", labels.shape)
    break

for eff_images, vit_images, labels in valid_loader:
    print("EfficientNET Image batch shape:", eff_images.shape)
    print("ViT Image batch shape:", vit_images.shape)
    print("Label batch shape:", labels.shape)
    break

EfficientNET Image batch shape: torch.Size([32, 3, 224, 224])
ViT Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 14])
EfficientNET Image batch shape: torch.Size([32, 3, 224, 224])
ViT Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32, 14])


In [6]:
import torch
import torch.nn as nn
import torchvision

class FocalLossMultiLabel(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        return torchvision.ops.sigmoid_focal_loss(
            inputs=inputs,
            targets=targets,
            alpha=self.alpha,
            gamma=self.gamma,
            reduction=self.reduction
        )

class logit_kd_loss(nn.Module):
    def __init__(self, alpha=0.5, T=2.0, focal_alpha=0.25, focal_gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.T = T
        # Initialize the hard label loss (Student vs Ground Truth)
        self.focal_loss = FocalLossMultiLabel(alpha=focal_alpha, gamma=focal_gamma, reduction='mean')
        # Initialize the soft label loss (Student vs Teacher)
        self.bce_logits = nn.BCEWithLogitsLoss(reduction='mean')

    def forward(self, student_logits, teacher_logits, labels):
        # 1. Hard Loss computation
        hard_loss = self.focal_loss(student_logits, labels)

        # 2. Soft Loss computation
        # Soften the teacher probabilities using Temperature
        soft_targets = torch.sigmoid(teacher_logits / self.T)
        # Scale the student logits to match
        scaled_student_logits = student_logits / self.T
        
        soft_loss = self.bce_logits(scaled_student_logits, soft_targets)

        # 3. Combine and apply T^2 gradient correction
        total_loss = (1.0 - self.alpha) * hard_loss + self.alpha * soft_loss * (self.T ** 2)
        
        return total_loss


class FeatureDistillationLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # The paper utilizes squared L2 distance:
        self.mse = nn.MSELoss(reduction='mean')

    def forward(self, pca_student_attn, teacher_attn, pca_student_v, teacher_v, gl_student_feats, teacher_feats):
        # L_proj1: PCA Loss (Aligns the Mixed Attention Maps and the Value Matrices)
        l_proj1_attn = self.mse(pca_student_attn, teacher_attn)
        l_proj1_v = self.mse(pca_student_v, teacher_v)
        l_proj1 = l_proj1_attn + l_proj1_v

        # L_proj2: GL Loss (Aligns the Group-wise Projected Feature Maps)
        l_proj2 = self.mse(gl_student_feats, teacher_feats)

        # Return the combined feature loss
        return l_proj1 + l_proj2

In [7]:
import timm
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import ViTForImageClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 1. INITIALIZE STUDENT (EfficientNet-B0)
# ==========================================
student_model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=14)
student_model.to(device)

# ==========================================
# 2. INITIALIZE TEACHER (Hugging Face ViT)
# ==========================================
# REPLACE with the actual dataset folder name from your Kaggle data pane
teacher_model_dir = '/kaggle/input/datasets/thageo/vit-full-label-weights/vit-chest-xray-full-labels-epoch-4' 
teacher_model = ViTForImageClassification.from_pretrained(teacher_model_dir)
teacher_model.to(device)

# Lock the teacher model gradients
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

# ==========================================
# 3. INITIALIZE PROJECTORS
# ==========================================
# EfficientNet-B0 Stage 4 (14x14 grid) has 112 channels. ViT-B/16 has 768 dimensions.
cnn_channels = 112 
vit_dim = 768

pca_projector = PCAProjector(cnn_channels=cnn_channels, vit_dim=vit_dim).to(device)
gl_projector = GLProjector(cnn_channels=cnn_channels, vit_dim=vit_dim).to(device)

# ==========================================
# 4. OPTIMIZER & SCHEDULER
# ==========================================
# Combine all trainable parameters
trainable_params = (
    list(student_model.parameters()) + 
    list(pca_projector.parameters()) + 
    list(gl_projector.parameters())
)

optimizer = AdamW(trainable_params, lr=1e-5) 
scheduler = CosineAnnealingLR(optimizer, T_max=10)

# ==========================================
# 5. LOSS FUNCTIONS
# ==========================================
# REPLACED logit_kd_loss to isolate hard labels
criterion_task = FocalLossMultiLabel(alpha=0.25, gamma=2.0)
criterion_feature = FeatureDistillationLoss()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

In [8]:
import gc
import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings

warnings.filterwarnings('ignore', category=UserWarning, module='sklearn.metrics')

def train_distillation_model(
    student_model, teacher_model, pca_projector, gl_projector, 
    train_loader, valid_loader, optimizer, scheduler, 
    criterion_task, criterion_feature, device, epochs=5, beta=1.0
):
    scaler = torch.amp.GradScaler()

    class_names = [
        'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 
        'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 
        'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices'
    ]

    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False

    # =======================================
    # 1. MANUAL HOOK SETUP FOR INTERCEPTION
    # =======================================
    activations = {}
    def get_activation(name):
        def hook(model, input, output):
            activations[name] = output
        return hook

    teacher_layer = teacher_model.vit.encoder.layer[-1].attention.attention
    h1 = teacher_layer.query.register_forward_hook(get_activation('teacher_q'))
    h2 = teacher_layer.key.register_forward_hook(get_activation('teacher_k'))
    h3 = teacher_layer.value.register_forward_hook(get_activation('teacher_v'))
    h4 = teacher_model.vit.layernorm.register_forward_hook(get_activation('teacher_feats'))

    h5 = student_model.blocks[4].register_forward_hook(get_activation('student_stage_4'))

    # =======================================
    # EPOCH LOOP
    # =======================================
    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch + 1}/{epochs} ---")

        student_model.train()
        pca_projector.train()
        gl_projector.train()
        
        # Initialize split tracking variables
        train_loss = 0.0
        train_task_loss = 0.0
        train_feat_loss = 0.0

        for effnet_images, vit_images, labels in train_loader:
            effnet_images = effnet_images.to(device)
            vit_images = vit_images.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                with torch.amp.autocast(device_type="cuda"):
                    teacher_logits = teacher_model(vit_images).logits

            with torch.amp.autocast(device_type="cuda"):
                student_logits = student_model(effnet_images)
                
                s_feat = activations['student_stage_4']

                # CRITICAL FIX: Add .contiguous() to prevent CUDA memory layout crashes
                t_q = activations['teacher_q'][:, 1:, :].contiguous()
                t_k = activations['teacher_k'][:, 1:, :].contiguous()
                t_v = activations['teacher_v'][:, 1:, :].contiguous()
                t_feat = activations['teacher_feats'][:, 1:, :].contiguous()

                scale = t_q.shape[-1] ** -0.5
                teacher_attn_map = torch.softmax((t_q @ t_k.transpose(-2, -1)) * scale, dim=-1)

                unused_pca_out, pca_attn_map, pca_v = pca_projector(s_feat, t_q, t_k, t_v, p=0.5)
                gl_s_feat = gl_projector(s_feat)

                # E. LOSS CALCULATION
                # REMOVED teacher_logits from the task loss
                loss_task = criterion_task(student_logits, labels)
                
                loss_feature = criterion_feature(
                    pca_attn_map, teacher_attn_map, 
                    pca_v, t_v, 
                    gl_s_feat, t_feat
                )
                
                total_loss = loss_task + (beta * loss_feature)

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            # Accumulate split losses
            train_loss += total_loss.item()
            train_task_loss += loss_task.item()
            train_feat_loss += loss_feature.item()

            activations.clear()
            del effnet_images, vit_images, labels, teacher_logits
            del student_logits, s_feat, t_q, t_k, t_v, t_feat, teacher_attn_map
            del unused_pca_out, pca_attn_map, pca_v, gl_s_feat, loss_task, loss_feature, total_loss

        # Average split losses
        train_loss /= len(train_loader)
        train_task_loss /= len(train_loader)
        train_feat_loss /= len(train_loader)
        
        print(f"Training Loss -> Total: {train_loss:.4f} | Task: {train_task_loss:.4f} | Feature: {train_feat_loss:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

        # =======================
        # VALIDATION PHASE
        # =======================
        student_model.eval()
        pca_projector.eval()
        gl_projector.eval()
        
        # Initialize validation split tracking
        valid_loss = 0.0
        valid_task_loss = 0.0
        valid_feat_loss = 0.0
        
        all_labels = []
        all_preds = []

        with torch.no_grad():
            for effnet_images, vit_images, labels in valid_loader:
                effnet_images = effnet_images.to(device)
                vit_images = vit_images.to(device)
                labels = labels.to(device)

                with torch.amp.autocast(device_type="cuda"):
                    teacher_logits = teacher_model(vit_images).logits
                    student_logits = student_model(effnet_images)
                    
                    s_feat = activations['student_stage_4']

                    # Apply contiguous fix to validation as well
                    t_q = activations['teacher_q'][:, 1:, :].contiguous()
                    t_k = activations['teacher_k'][:, 1:, :].contiguous()
                    t_v = activations['teacher_v'][:, 1:, :].contiguous()
                    t_feat = activations['teacher_feats'][:, 1:, :].contiguous()

                    scale = t_q.shape[-1] ** -0.5
                    teacher_attn_map = torch.softmax((t_q @ t_k.transpose(-2, -1)) * scale, dim=-1)

                    unused_pca_out, pca_attn_map, pca_v = pca_projector(s_feat, t_q, t_k, t_v, p=0.5)
                    gl_s_feat = gl_projector(s_feat)

# E. LOSS CALCULATION
                    loss_task = criterion_task(student_logits, labels)
                    
                    loss_feature = criterion_feature(
                        pca_attn_map, teacher_attn_map, 
                        pca_v, t_v, 
                        gl_s_feat, t_feat
                    )
                    
                    total_loss = loss_task + (beta * loss_feature)

                # Accumulate validation split losses
                valid_loss += total_loss.item()
                valid_task_loss += loss_task.item()
                # FIXED: Changed loss_feat to loss_feature
                valid_feat_loss += loss_feature.item() 
                
                probs = torch.sigmoid(student_logits)
                
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs.cpu().numpy())
                
                activations.clear()
                del effnet_images, vit_images, labels, teacher_logits, teacher_attn_map
                del student_logits, s_feat, t_q, t_k, t_v, t_feat
                # FIXED: Changed loss_feat to loss_feature in the delete block as well
                del unused_pca_out, pca_attn_map, pca_v, gl_s_feat, loss_task, loss_feature, total_loss, probs
        # Average validation split losses
        valid_loss /= len(valid_loader)
        valid_task_loss /= len(valid_loader)
        valid_feat_loss /= len(valid_loader)
        
        all_labels = np.vstack(all_labels)
        all_preds = np.vstack(all_preds)

        # =======================
        # ROBUST METRICS CALCULATION
        # =======================
        per_class_roc_auc = []
        per_class_ap = []
        
        for i in range(len(class_names)):
            if len(np.unique(all_labels[:, i])) > 1:
                roc_score = roc_auc_score(all_labels[:, i], all_preds[:, i])
                ap_score = average_precision_score(all_labels[:, i], all_preds[:, i])
            else:
                roc_score = np.nan
                ap_score = np.nan
                
            per_class_roc_auc.append(roc_score)
            per_class_ap.append(ap_score)

        valid_roc_scores = [s for s in per_class_roc_auc if not np.isnan(s)]
        valid_ap_scores = [s for s in per_class_ap if not np.isnan(s)]
        
        macro_roc_auc = np.mean(valid_roc_scores) if valid_roc_scores else np.nan
        macro_ap = np.mean(valid_ap_scores) if valid_ap_scores else np.nan

        del all_labels, all_preds
        gc.collect()
        
        # Print updated validation breakdown
        print(f"Validation Loss -> Total: {valid_loss:.4f} | Task: {valid_task_loss:.4f} | Feature: {valid_feat_loss:.4f}")
        print(f"Macro ROC-AUC: {macro_roc_auc:.4f} | Macro Avg Precision: {macro_ap:.4f}")
        
        print(f"{'Class':<28} | {'ROC-AUC':<8} | {'Avg Precision':<13}")
        print("-" * 55)
        for i, name in enumerate(class_names):
            roc_str = f"{per_class_roc_auc[i]:.4f}" if not np.isnan(per_class_roc_auc[i]) else "NaN"
            ap_str = f"{per_class_ap[i]:.4f}" if not np.isnan(per_class_ap[i]) else "NaN"
            print(f"{name:<28} | {roc_str:<8} | {ap_str:<13}")

        if scheduler is not None:
            scheduler.step()

        save_path = f"/kaggle/working/feature-hard-labels-kd-efficientnet-b0-epoch-{epoch+1}.pth"
        print(f"\nSaving checkpoint to {save_path}...")
        torch.save(student_model.state_dict(), save_path)

    # =======================================
    # TEARDOWN
    # =======================================
    h1.remove()
    h2.remove()
    h3.remove()
    h4.remove()
    h5.remove()

In [9]:
train_distillation_model(
    student_model=student_model, 
    teacher_model=teacher_model, 
    pca_projector=pca_projector,    # New addition
    gl_projector=gl_projector,      # New addition
    train_loader=train_loader, 
    valid_loader=valid_loader, 
    optimizer=optimizer,
    scheduler=scheduler,
    criterion_task=criterion_task,       # Renamed
    criterion_feature=criterion_feature, # New addition
    device=device, 
    epochs=5,
    beta=0.01  # This was dropped down from 1 so that the classification head actually learns to classify. With beta =1 we would get feature really close to the ViT but we wouldn't know what to do with them 
)


--- Epoch 1/5 ---
Training Loss -> Total: 0.1119 | Task: 0.0447 | Feature: 6.7122
Validation Loss -> Total: 0.0660 | Task: 0.0421 | Feature: 2.3883
Macro ROC-AUC: 0.7355 | Macro Avg Precision: 0.4419
Class                        | ROC-AUC  | Avg Precision
-------------------------------------------------------
No Finding                   | 0.8841   | 0.5918       
Enlarged Cardiomediastinum   | 0.6214   | 0.5660       
Cardiomegaly                 | 0.8005   | 0.6034       
Lung Opacity                 | 0.8317   | 0.8391       
Lung Lesion                  | 0.7167   | 0.0149       
Edema                        | 0.8567   | 0.6233       
Consolidation                | 0.7870   | 0.3361       
Pneumonia                    | 0.5171   | 0.0425       
Atelectasis                  | 0.7465   | 0.5721       
Pneumothorax                 | 0.5542   | 0.1215       
Pleural Effusion             | 0.8493   | 0.7010       
Pleural Other                | 0.6159   | 0.0110       
Fracture       